[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/patterns/plain_python_patterns.ipynb)

# Seven agentic patterns in plain Python

All examples use the same bounded household-food-waste catalogue and deterministic mock model. Prompts, structured decisions, execution, state, traces and stopping rules remain visible. Expected runtime is under one minute on a normal CPU; no credentials or model download are required.

In [ ]:
# Colab dependency pin; uv.lock is authoritative in a cloned repository.
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydantic==2.12.5"], check=True)

In [ ]:
import asyncio
import json
from pathlib import Path
from typing import ClassVar

from pydantic import BaseModel, ConfigDict, Field

ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            "feature/notebook-rebuild",
            "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
            str(ROOT),
        ],
        check=True,
    )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))

from agentic_tutorial.models import DeterministicMockClient  # noqa: E402
from agentic_tutorial.models.providers.fixtures import ScriptedScenarioFixture  # noqa: E402
from agentic_tutorial.schemas import (  # noqa: E402
    CriticDecision,
    Message,
    MessageRole,
    PlanDecision,
    RouteDecision,
    ToolDefinition,
)

## Shared task, model fixtures, tools and schemas

The model proposes typed decisions. `search_catalogue` and the surrounding Python validate and execute them. Retrieved records are data, never instructions.

In [ ]:
class StrictModel(BaseModel):
    model_config = ConfigDict(extra="forbid")


class ClaimProposal(StrictModel):
    schema_id: ClassVar[str] = "tutorial.claim.v1"
    claim: str = Field(min_length=1)
    source_id: str = Field(min_length=1)
    supported: bool


class PatternAnswer(StrictModel):
    schema_id: ClassVar[str] = "tutorial.pattern_answer.v1"
    answer: str = Field(min_length=1)
    source_ids: tuple[str, ...]


class ParallelDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.parallel.v1"
    queries: tuple[str, ...] = Field(min_length=2, max_length=3)
    aggregation: str = Field(pattern="^validated_union$")


class SearchArguments(StrictModel):
    query: str = Field(min_length=1)
    max_results: int = Field(ge=1, le=5)


class WorkerAssignment(StrictModel):
    worker: str = Field(pattern="^(intervention_reviewer|planning_reviewer)$")
    query: str = Field(min_length=1)


class OrchestrationDecision(StrictModel):
    schema_id: ClassVar[str] = "tutorial.orchestration.v1"
    assignments: tuple[WorkerAssignment, ...] = Field(min_length=1, max_length=2)


OrchestrationDecision.model_rebuild(_types_namespace={"WorkerAssignment": WorkerAssignment})


fixture_data = json.loads(
    (ROOT / "data/research_assistant/pattern_mock_scenarios.json").read_text()
)
catalogue = fixture_data["catalogue"]
known_source_ids = {record["source_id"] for record in catalogue}
task = "Which interventions reduce household food waste, and under what conditions?"


def model_for(scenario: str) -> DeterministicMockClient:
    fixture = ScriptedScenarioFixture.model_validate(
        {
            "fixture_version": fixture_data["fixture_version"],
            "scenario": scenario,
            "provenance": fixture_data["provenance"],
            "responses": fixture_data["scenarios"][scenario],
        }
    )
    return DeterministicMockClient(fixture)


def search_catalogue(arguments: SearchArguments) -> list[dict]:
    query_terms = set(arguments.query.casefold().split())
    ranked = [
        record
        for record in catalogue
        if query_terms & set(" ".join(record["topics"]).casefold().split())
    ]
    return ranked[: arguments.max_results]


search_tool = ToolDefinition(
    name="search_catalogue",
    description="Search the bounded food-waste catalogue.",
    parameters=SearchArguments.model_json_schema(),
)


def user(text: str) -> Message:
    return Message(role=MessageRole.USER, content=text)

## 1. Prompt chaining

Fixed flow: extract → validate → synthesise. Limitation: an extraction error propagates unless stopped at the validation boundary.

In [ ]:
chain_model = model_for("prompt_chaining")
chain_state = {"trace": [], "termination": None}
claim_response = await chain_model.generate(
    [user("Extract one supported claim from food-waste-001.")], response_schema=ClaimProposal
)
claim = ClaimProposal.model_validate(claim_response.structured_output)
claim_valid = claim.supported and claim.source_id in known_source_ids
chain_state["trace"].append(
    {"stage": "extract", "decision": claim.model_dump(), "valid": claim_valid}
)
if claim_valid:
    answer_response = await chain_model.generate(
        [user(f"Synthesise only this validated claim: {claim.model_dump()}")],
        response_schema=PatternAnswer,
    )
    chain_answer = PatternAnswer.model_validate(answer_response.structured_output)
    citations_valid = set(chain_answer.source_ids) == {claim.source_id}
    chain_state["trace"].append(
        {"stage": "synthesise", "decision": chain_answer.model_dump(), "valid": citations_valid}
    )
    chain_state["termination"] = "criteria_met" if citations_valid else "invalid_citation"
else:
    chain_state["termination"] = "invalid_claim"
chain_state

## 2. Routing

A model selects one enumerated route; Python rejects unknown routes and executes the selected handler. Limitation: a semantically wrong but valid route can still misdirect work.

In [ ]:
route_model = model_for("routing")
route_response = await route_model.generate(
    [user(f"Route this request: {task}")], response_schema=RouteDecision
)
route = RouteDecision.model_validate(route_response.structured_output)
handlers = {
    "research": lambda: search_catalogue(
        SearchArguments(query="meal planning portion", max_results=3)
    ),
    "clarify": lambda: [],
}
route_valid = route.route in handlers
route_result = handlers[route.route]() if route_valid else []
route_state = {
    "decision": route.model_dump(),
    "result_count": len(route_result),
    "trace": ["route_validated", f"handler:{route.route}"] if route_valid else ["invalid_route"],
    "termination": "criteria_met" if route_valid else "safe_fallback",
}
route_state

## 3. Parallelisation

The model proposes independent queries; validated branches run concurrently and deterministic aggregation removes duplicates. Limitation: parallel branches can duplicate effort or conflict.

In [ ]:
parallel_model = model_for("parallelisation")
parallel_response = await parallel_model.generate(
    [user(f"Propose two independent searches for: {task}")], response_schema=ParallelDecision
)
parallel_decision = ParallelDecision.model_validate(parallel_response.structured_output)


async def search_branch(query: str) -> list[dict]:
    return search_catalogue(SearchArguments(query=query, max_results=2))


branch_results = await asyncio.gather(
    *(search_branch(query) for query in parallel_decision.queries)
)
parallel_records = {record["source_id"]: record for branch in branch_results for record in branch}
parallel_state = {
    "decision": parallel_decision.model_dump(),
    "branch_counts": [len(branch) for branch in branch_results],
    "source_ids": sorted(parallel_records),
    "trace": ["branches_validated", "branches_completed", "validated_union"],
    "termination": "criteria_met" if parallel_records else "empty_results",
}
parallel_state

## 4. ReAct-style tool use

A bounded decision → validated action → observation loop. Free-form output never executes a tool. Limitation: untrusted observations and repeated calls can redirect or exhaust the loop.

In [ ]:
react_model = model_for("react")
react_state = {
    "observations": [],
    "trace": [],
    "answer": None,
    "termination": "step_budget_exhausted",
}
for step in range(1, 4):
    if not react_state["observations"]:
        response = await react_model.generate([user(task)], tools=[search_tool])
        if len(response.tool_calls) != 1 or response.tool_calls[0].name != search_tool.name:
            react_state["termination"] = "invalid_tool"
            break
        arguments = SearchArguments.model_validate(response.tool_calls[0].arguments)
        observation = search_catalogue(arguments)
        react_state["observations"].append(observation)
        react_state["trace"].append(
            {
                "step": step,
                "action": response.tool_calls[0].model_dump(),
                "observation_count": len(observation),
            }
        )
    else:
        response = await react_model.generate(
            [user(f"Answer from this observation: {react_state['observations']}")],
            response_schema=PatternAnswer,
        )
        answer = PatternAnswer.model_validate(response.structured_output)
        if set(answer.source_ids).issubset(known_source_ids):
            react_state["answer"] = answer.model_dump()
            react_state["termination"] = "criteria_met"
        else:
            react_state["termination"] = "invalid_citation"
        react_state["trace"].append(
            {"step": step, "decision": "finish", "valid": react_state["answer"] is not None}
        )
        break
react_state

## 5. Planner–executor

The model proposes a typed dependency plan; the executor authorises known steps only. Limitation: a valid-looking plan may still be infeasible or stale.

In [ ]:
planner_model = model_for("planner_executor")
plan_response = await planner_model.generate(
    [user(f"Create a dependency-aware plan for: {task}")], response_schema=PlanDecision
)
plan = PlanDecision.model_validate(plan_response.structured_output)
allowed_steps = {"search", "validate", "report"}
completed, plan_trace = set(), []
for step in plan.steps:
    ready = step.step_id in allowed_steps and set(step.depends_on).issubset(completed)
    plan_trace.append({"step_id": step.step_id, "ready": ready})
    if not ready:
        break
    completed.add(step.step_id)
plan_state = {
    "plan": plan.model_dump(),
    "completed": sorted(completed),
    "trace": plan_trace,
    "termination": "criteria_met" if completed == allowed_steps else "invalid_plan",
}
plan_state

## 6. Critic–reviser

Critique is grounded in source and citation criteria and revision is limited to one attempt. Limitation: model self-critique is not independent verification.

In [ ]:
critic_model = model_for("critic_reviser")
draft = PatternAnswer.model_validate(
    (
        await critic_model.generate([user("Draft an answer.")], response_schema=PatternAnswer)
    ).structured_output
)
critique = CriticDecision.model_validate(
    (
        await critic_model.generate(
            [user(f"Check support and conditions for: {draft.model_dump()}")],
            response_schema=CriticDecision,
        )
    ).structured_output
)
critic_trace = [
    {"stage": "draft", "value": draft.model_dump()},
    {"stage": "critique", "value": critique.model_dump()},
]
final_answer = draft
if not critique.accepted:
    final_answer = PatternAnswer.model_validate(
        (
            await critic_model.generate(
                [user(f"Revise once using catalogue evidence; issues: {critique.issues}")],
                response_schema=PatternAnswer,
            )
        ).structured_output
    )
    critic_trace.append({"stage": "revision", "value": final_answer.model_dump()})
critic_valid = bool(final_answer.source_ids) and set(final_answer.source_ids).issubset(
    known_source_ids
)
critic_state = {
    "answer": final_answer.model_dump(),
    "revisions": 1 if not critique.accepted else 0,
    "trace": critic_trace,
    "termination": "criteria_met" if critic_valid else "revision_budget_exhausted",
}
critic_state

## 7. Orchestrator–worker

The orchestrator proposes bounded assignments; named workers receive only the read-only search operation and Python aggregates their outputs. Limitation: delegation adds coordination overhead and can produce inconsistent findings.

In [ ]:
orchestrator_model = model_for("orchestrator_worker")
orchestration = OrchestrationDecision.model_validate(
    (
        await orchestrator_model.generate(
            [user(f"Assign bounded specialist searches for: {task}")],
            response_schema=OrchestrationDecision,
        )
    ).structured_output
)


async def run_worker(assignment: WorkerAssignment) -> dict:
    records = search_catalogue(SearchArguments(query=assignment.query, max_results=2))
    return {"worker": assignment.worker, "query": assignment.query, "records": records}


worker_results = await asyncio.gather(
    *(run_worker(assignment) for assignment in orchestration.assignments)
)
worker_source_ids = {
    record["source_id"] for result in worker_results for record in result["records"]
}
orchestrated_answer = PatternAnswer.model_validate(
    (
        await orchestrator_model.generate(
            [user(f"Synthesise these worker results: {worker_results}")],
            response_schema=PatternAnswer,
        )
    ).structured_output
)
orchestration_valid = set(orchestrated_answer.source_ids) == worker_source_ids
orchestrator_state = {
    "decision": orchestration.model_dump(),
    "worker_results": worker_results,
    "answer": orchestrated_answer.model_dump(),
    "trace": ["assignments_validated", "workers_completed", "synthesis_validated"],
    "termination": "criteria_met" if orchestration_valid else "invalid_aggregation",
}
orchestrator_state

## Evaluation summary

Every example must terminate explicitly, preserve an operational trace and satisfy its pattern-specific acceptance check. Repeated mock runs are byte-stable because all model proposals and catalogue records are versioned fixtures.

In [ ]:
pattern_evaluations = {
    "prompt_chaining": chain_state["termination"] == "criteria_met"
    and len(chain_state["trace"]) == 2,
    "routing": route_state["termination"] == "criteria_met" and route_state["result_count"] == 2,
    "parallelisation": parallel_state["termination"] == "criteria_met"
    and parallel_state["source_ids"] == ["food-waste-001", "food-waste-002"],
    "react": react_state["termination"] == "criteria_met" and len(react_state["trace"]) == 2,
    "planner_executor": plan_state["termination"] == "criteria_met",
    "critic_reviser": critic_state["termination"] == "criteria_met"
    and critic_state["revisions"] == 1,
    "orchestrator_worker": orchestrator_state["termination"] == "criteria_met"
    and len(worker_results) == 2,
}
failure_demonstrations = {
    "invalid_route": "safe fallback",
    "invalid_tool": "no execution",
    "invalid_plan": "dependency stop",
    "revision_limit": 1,
}
assert all(pattern_evaluations.values()), pattern_evaluations
{"evaluation": pattern_evaluations, "failures": failure_demonstrations, "model_provider": "mock"}